# Onshape Pack and Go
Export all parts (STEP) and their linked drawings (PDF) from an Onshape assembly.

**On Google Colab:**
1. Click the 🔑 **Secrets** tab in the left sidebar
2. Add `ONSHAPE_ACCESS_KEY` and `ONSHAPE_SECRET_KEY` (from [Onshape Developer Portal](https://dev-portal.onshape.com/))
3. Paste your assembly URL in the Config cell below
4. Runtime → Run all

**Running locally:**
1. Fill in your keys in the `.env` file next to this notebook
2. Paste your assembly URL in the Config cell below
3. Run all cells

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install requests python-dotenv --quiet

In [ ]:
# ── 2. Configuration — fill in your assembly URL here ───────────────────────
ASSEMBLY_URL = "https://cad.onshape.com/documents/YOUR_DOCUMENT_ID/w/YOUR_WORKSPACE_ID/e/YOUR_ELEMENT_ID"

In [ ]:
# ── 3. Load API keys ─────────────────────────────────────────────────────────
import os

try:
    from google.colab import userdata
    ACCESS_KEY = userdata.get('ONSHAPE_ACCESS_KEY')
    SECRET_KEY = userdata.get('ONSHAPE_SECRET_KEY')
    print("✓ API keys loaded from Colab Secrets")
except Exception:
    # Local: load from .env file if present
    from dotenv import load_dotenv
    load_dotenv()
    ACCESS_KEY = os.environ.get('ONSHAPE_ACCESS_KEY', '')
    SECRET_KEY = os.environ.get('ONSHAPE_SECRET_KEY', '')
    if ACCESS_KEY:
        print("✓ API keys loaded from .env file")
    else:
        print("Running locally — keys not found in .env")

if not ACCESS_KEY or not SECRET_KEY:
    raise ValueError("API keys not found. Add ONSHAPE_ACCESS_KEY and ONSHAPE_SECRET_KEY to Colab Secrets or a .env file.")

In [ ]:
# ── 4. Onshape API client with HMAC-SHA256 auth ──────────────────────────────
import hashlib
import hmac
import base64
import random
import string
import time
from datetime import datetime, timezone
from email.utils import formatdate
import requests

BASE_URL = "https://cad.onshape.com"


def _make_nonce(length=25):
    return ''.join(random.choices(string.ascii_letters + string.digits, k=length))


def _onshape_request(method, path, query=None, body=None):
    """Make an authenticated Onshape API request."""
    query = query or {}
    nonce = _make_nonce()
    date = formatdate(usegmt=True)
    content_type = "application/json" if body else ""

    # Build query string (sorted for consistent signing)
    query_str = "&".join(f"{k}={v}" for k, v in sorted(query.items()))
    full_path = f"/api/v6{path}"

    # String to sign (all lowercase)
    string_to_sign = "\n".join([
        method.lower(),
        nonce.lower(),
        date.lower(),
        content_type.lower(),
        full_path.lower(),
        query_str.lower(),
        ""
    ])

    signature = base64.b64encode(
        hmac.new(SECRET_KEY.encode(), string_to_sign.encode(), hashlib.sha256).digest()
    ).decode()

    auth = f"On {ACCESS_KEY}:HmacSHA256:{signature}"

    headers = {
        "Authorization": auth,
        "Date": date,
        "On-Nonce": nonce,
        "Accept": "application/json",
    }
    if content_type:
        headers["Content-Type"] = content_type

    url = BASE_URL + full_path
    if query_str:
        url += "?" + query_str

    resp = requests.request(method, url, headers=headers, json=body)
    resp.raise_for_status()
    return resp


def api_get(path, query=None):
    return _onshape_request("GET", path, query=query).json()


def api_post(path, body=None, query=None):
    return _onshape_request("POST", path, query=query, body=body).json()


def api_get_binary(path, query=None):
    return _onshape_request("GET", path, query=query).content


print("✓ Onshape client ready")

In [ ]:
# ── 5. Parse assembly URL ────────────────────────────────────────────────────
import re

def parse_onshape_url(url):
    """Extract document ID, workspace/version/microversion type+ID, and element ID."""
    pattern = r"documents/([a-f0-9]+)/(w|v|m)/([a-f0-9]+)/e/([a-f0-9]+)"
    m = re.search(pattern, url)
    if not m:
        raise ValueError(f"Could not parse Onshape URL: {url}")
    did, wvm, wvmid, eid = m.group(1), m.group(2), m.group(3), m.group(4)
    return did, wvm, wvmid, eid


did, wvm, wvmid, eid = parse_onshape_url(ASSEMBLY_URL)
print(f"Document:  {did}")
print(f"Workspace: {wvmid} ({wvm})")
print(f"Element:   {eid}")

In [ ]:
# ── 6. Get all unique RELEASED parts from the assembly ───────────────────────

def get_assembly_parts(did, wvm, wvmid, eid):
    """
    Walk the assembly BOM to collect all unique parts in the "Released" state.
    Returns a list of dicts with keys: partId, elementId, documentId, documentMicroversion, name.
    """
    data = api_get(
        f"/assemblies/d/{did}/{wvm}/{wvmid}/e/{eid}/bom",
        query={"bomType": "flattened", "indented": "false", "multiLevel": "false"},
    )

    # Find the index of the "State" column in the BOM headers
    headers = data.get("headers", [])
    state_col = next(
        (h["id"] for h in headers if h.get("name", "").lower() == "state"),
        None,
    )
    if state_col is None:
        print("⚠ No 'State' column found in BOM — release filtering skipped.")

    seen = set()
    parts = []
    skipped = []

    for row in data.get("rows", []):
        item = row.get("item", {})

        # Check release state
        if state_col is not None:
            props = {p["columnId"]: p.get("value", "") for p in item.get("properties", [])}
            state = props.get(state_col, "")
            if state.lower() != "released":
                skipped.append(item.get("name", "?") + f" ({state or 'no state'})")
                continue

        sources = item.get("itemSources", [])
        if not sources:
            continue
        src = sources[0]
        part_id = src.get("partId")
        element_id = src.get("elementId")
        doc_id = src.get("documentId") or did
        doc_microversion = src.get("documentMicroversion", "")
        name = item.get("name", f"part_{part_id}")

        key = (doc_id, element_id, part_id)
        if key in seen:
            continue
        seen.add(key)
        parts.append({
            "partId": part_id,
            "elementId": element_id,
            "documentId": doc_id,
            "documentMicroversion": doc_microversion,
            "name": name,
        })

    if skipped:
        print(f"Skipped {len(skipped)} non-released part(s):")
        for s in skipped:
            print(f"  ✗ {s}")

    return parts


parts = get_assembly_parts(did, wvm, wvmid, eid)
print(f"\nFound {len(parts)} released part(s):")
for p in parts:
    print(f"  • {p['name']} (partId={p['partId']}, elementId={p['elementId']})")

In [ ]:
# ── 7. Export each part as STEP ──────────────────────────────────────────────
import time

POLL_INTERVAL = 3   # seconds between status checks
POLL_TIMEOUT  = 300 # seconds before giving up on a translation


def export_part_step(part):
    """
    Kick off a STEP translation for a single part and return the file bytes.
    Uses the Onshape translation API (async with polling).
    """
    p_did = part["documentId"]
    p_eid = part["elementId"]
    part_id = part["partId"]

    # Start translation
    body = {
        "format": "STEP",
        "partIds": part_id,
        "storeInDocument": False,
    }
    result = api_post(
        f"/parts/d/{p_did}/{wvm}/{wvmid}/e/{p_eid}/partid/{part_id}/export",
        body=body,
    )
    # For inline exports Onshape returns the file directly via a redirect;
    # for async it returns a translation ID. Handle both.
    if "href" in result:
        # Async translation — poll until done
        translation_id = result["id"]
        return _poll_translation(translation_id)
    # Inline response (shouldn't happen for STEP, but just in case)
    return result


def _poll_translation(translation_id):
    """Poll a translation job until complete, then download the result."""
    deadline = time.time() + POLL_TIMEOUT
    while time.time() < deadline:
        status = api_get(f"/translations/{translation_id}")
        state = status.get("requestState", "")
        if state == "DONE":
            ext_data_ids = status.get("resultExternalDataIds", [])
            if not ext_data_ids:
                raise RuntimeError(f"Translation {translation_id} finished but returned no files.")
            # Download the first (and usually only) result file
            doc_id = status["documentId"]
            ext_id = ext_data_ids[0]
            return api_get_binary(f"/documents/d/{doc_id}/externaldata/{ext_id}")
        elif state == "FAILED":
            raise RuntimeError(f"Translation {translation_id} failed: {status.get('failureReason')}")
        time.sleep(POLL_INTERVAL)
    raise TimeoutError(f"Translation {translation_id} did not complete within {POLL_TIMEOUT}s")


step_files = {}  # name → bytes

for part in parts:
    print(f"Exporting STEP: {part['name']} ...", end=" ", flush=True)
    try:
        data = export_part_step(part)
        safe_name = re.sub(r'[^\w\-.]', '_', part['name'])
        step_files[f"{safe_name}.step"] = data
        print("✓")
    except Exception as e:
        print(f"✗ ({e})")

print(f"\nExported {len(step_files)} STEP file(s).")

In [ ]:
# ── 8. Find drawings explicitly linked to assembly parts ─────────────────────

def get_document_elements(doc_id, wvm, wvmid):
    """List all elements in a document."""
    return api_get(f"/documents/d/{doc_id}/{wvm}/{wvmid}/elements")


def get_drawing_refs(doc_id, wvm, wvmid, drawing_eid):
    """
    Return the list of refs that a drawing explicitly links to.
    Each ref is a dict with documentId, elementId, partId.
    """
    refs = []
    try:
        data = api_get(
            f"/drawings/d/{doc_id}/{wvm}/{wvmid}/e/{drawing_eid}/renderspec"
        )
        for item in data.get("documentRenderSpecs", []):
            part_specs = item.get("partRenderSpecs", [])
            if part_specs:
                for part_spec in part_specs:
                    refs.append({
                        "documentId": item.get("documentId", ""),
                        "elementId": item.get("elementId", ""),
                        "partId": part_spec.get("partId", ""),
                    })
            else:
                # Drawing references a whole element (e.g. assembly), no specific partId
                refs.append({
                    "documentId": item.get("documentId", ""),
                    "elementId": item.get("elementId", ""),
                    "partId": "",
                })
    except Exception:
        pass
    return refs


def find_linked_drawings_in_document(scan_did, scan_wvm, scan_wvmid, part_keys, part_element_keys, parts):
    """
    Scan all drawing elements in a document and return those linked to any of the given parts.
    Returns a list of (element, matched_part_names, doc_id, wvm, wvmid) tuples.
    """
    try:
        elements = get_document_elements(scan_did, scan_wvm, scan_wvmid)
    except Exception as e:
        print(f"  ⚠ Could not list elements for document {scan_did}: {e}")
        return []

    drawing_elements = [el for el in elements if el.get("type") == "DRAWING"]
    linked = []

    for el in drawing_elements:
        d_eid = el["id"]
        d_name = el.get("name", d_eid)
        refs = get_drawing_refs(scan_did, scan_wvm, scan_wvmid, d_eid)
        matched_parts = []
        for ref in refs:
            key3 = (ref["documentId"], ref["elementId"], ref["partId"])
            key2 = (ref["documentId"], ref["elementId"])
            for p in parts:
                if (p["documentId"], p["elementId"], p["partId"]) == key3:
                    matched_parts.append(p["name"])
                elif (p["documentId"], p["elementId"]) == key2 and not ref["partId"]:
                    matched_parts.append(p["name"])
        if matched_parts:
            linked.append((el, matched_parts, scan_did, scan_wvm, scan_wvmid))
            print(f"  ✓ '{d_name}' → linked to: {', '.join(set(matched_parts))}")

    return linked


# Build lookup sets
part_keys = {
    (p["documentId"], p["elementId"], p["partId"]) for p in parts
}
part_element_keys = {
    (p["documentId"], p["elementId"]) for p in parts
}
part_element_keys.add((did, eid))  # include the assembly element itself

# Collect all unique documents to scan: the assembly doc + any external part docs
docs_to_scan = {did: (wvm, wvmid)}
for p in parts:
    p_did = p["documentId"]
    if p_did not in docs_to_scan:
        # External document — use the same wvm type but with that doc's workspace.
        # We use the microversion if available, otherwise fall back to the assembly workspace.
        if p.get("documentMicroversion"):
            docs_to_scan[p_did] = ("m", p["documentMicroversion"])
        else:
            # Try to find the main workspace of the external document
            try:
                doc_info = api_get(f"/documents/{p_did}")
                ext_wid = doc_info.get("defaultWorkspace", {}).get("id", "")
                if ext_wid:
                    docs_to_scan[p_did] = ("w", ext_wid)
            except Exception:
                docs_to_scan[p_did] = (wvm, wvmid)  # fallback, likely wrong but won't crash

linked_drawings = []

for scan_did, (scan_wvm, scan_wvmid) in docs_to_scan.items():
    label = "assembly document" if scan_did == did else f"external document {scan_did}"
    print(f"\nScanning {label} for linked drawings...")
    found = find_linked_drawings_in_document(
        scan_did, scan_wvm, scan_wvmid, part_keys, part_element_keys, parts
    )
    linked_drawings.extend(found)

print(f"\n{len(linked_drawings)} total drawing(s) linked to released parts.")

In [ ]:
# ── 9. Export linked drawings as PDF ─────────────────────────────────────────

def export_drawing_pdf(doc_id, wvm, wvmid, drawing_eid):
    """Export a drawing element as PDF bytes."""
    body = {
        "format": "PDF",
        "storeInDocument": False,
    }
    result = api_post(
        f"/drawings/d/{doc_id}/{wvm}/{wvmid}/e/{drawing_eid}/export",
        body=body,
    )
    if "id" in result:
        return _poll_translation(result["id"])
    return result


pdf_files = {}  # name → bytes

for el, matched_parts, d_doc_id, d_wvm, d_wvmid in linked_drawings:
    d_eid = el["id"]
    d_name = el.get("name", d_eid)
    print(f"Exporting PDF: {d_name} ...", end=" ", flush=True)
    try:
        data = export_drawing_pdf(d_doc_id, d_wvm, d_wvmid, d_eid)
        safe_name = re.sub(r'[^\w\-.]', '_', d_name)
        pdf_files[f"{safe_name}.pdf"] = data
        print("✓")
    except Exception as e:
        print(f"✗ ({e})")

print(f"\nExported {len(pdf_files)} PDF file(s).")

In [ ]:
# ── 10. Package everything into a ZIP and download ───────────────────────────
import zipfile
import io

zip_name = "onshape_pack_and_go.zip"
zip_buffer = io.BytesIO()

with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    for filename, data in step_files.items():
        zf.writestr(f"parts/{filename}", data)
    for filename, data in pdf_files.items():
        zf.writestr(f"drawings/{filename}", data)

zip_bytes = zip_buffer.getvalue()
print(f"ZIP created: {len(zip_bytes) / 1024:.1f} KB")
print(f"  parts/    — {len(step_files)} STEP file(s)")
print(f"  drawings/ — {len(pdf_files)} PDF file(s)")

# Download in Colab
try:
    from google.colab import files
    files.download_bytes(zip_name, zip_bytes)
    print(f"\n✓ Download started: {zip_name}")
except ImportError:
    # Running locally — save to disk
    with open(zip_name, "wb") as f:
        f.write(zip_bytes)
    print(f"\n✓ Saved locally: {zip_name}")